# Sage Lattice Category Spike — Demonstration Notebook

The base parity spike replaces Sage's scattered lattice surfaces with one
owned category-shaped package. A lattice is a based projective $R$-module of
finite rank with an exact symmetric bilinear form — no ambient space, no
echelon coordinate frame.

**Prerequisite:** run `direnv allow` at the repository root so `.envrc`
prepends `computations/experiments` to `PYTHONPATH`. Then this notebook
imports the spike as `sage_lattice_category_spike` from any working
directory — no `sys.path` manipulation needed.

In [ ]:
from sage_lattice_category_spike.lattice_categories import (
    Lattice, RootLattice, U, SyntheticLatticeFromGram,
    Lattices, DiscriminantForms,
    SyntheticLattice, SyntheticLatticeElement,
    SyntheticBilinearDiscriminantForm, SyntheticQuadraticDiscriminantForm,
    SyntheticSourcedDiscriminantForm, SyntheticGenus,
    TorsionQuadraticForm, SyntheticDiscriminantSubgroup,
    IntegralLatticeGluing,
)
from sage.all import QQ, ZZ, matrix, identity_matrix, block_diagonal_matrix, vector

print('Spike loaded. Sage version:', sage.version.version)

## 1. Constructors

The section-1.4 `Lattices(R).from_gram_matrix` is the ONLY public entry into
the language. Named section-6 constructors (`Lattice`, `U`, `RootLattice`,
`IntegralLatticeGluing`) route through it.

#### String shortcuts
- `Lattice("A2")` → the standard Cartan Gram of $A_2$
- `Lattice("U")` → hyperbolic plane $\begin{pmatrix}0&1\\1&0\end{pmatrix}$
- `Lattice("D4")`, `Lattice("E8")`
- `Lattice("A2(-1)")` for the negative-definite convention (K3)

In [ ]:
# Section 6 constructors: string shortcuts route through the category entry
A2 = Lattice('A2', label='A2')
U = Lattice('U', label='U')
E8 = Lattice('E8', label='E8')
print(f'A₂: {A2}')
print(f'  Gram: {A2.gram_matrix()}')
print(f'  rank={A2.rank()}, det={A2.determinant()}, sig={A2.signature_pair()}')
print()
print(f'Hyperbolic plane: {U}')
print(f'  Gram: {U.gram_matrix()}')
print(f'  sig={U.signature_pair()}')
print()
print(f'E₈: {E8}')
print(f'  rank={E8.rank()}, det={E8.determinant()}, sig={E8.signature_pair()}')

# Mathematical assertions
assert A2.gram_matrix() == matrix(QQ, [[2, -1], [-1, 2]])  # standard Cartan Gram of A₂
assert U.gram_matrix() == matrix(QQ, [[0, 1], [1, 0]])       # hyperbolic plane
assert E8.rank() == 8 and E8.determinant() == 1              # E₈ is unimodular

In [ ]:
# Category membership: computed from the Gram at construction time
# Axiom gating: a predicate on the grammar determines which vocabulary resolves
print('A2 in Lattices(ZZ).PositiveDefinite():', A2 in Lattices(ZZ).PositiveDefinite())
print('U  in Lattices(ZZ).Indefinite():',       U  in Lattices(ZZ).Indefinite())
print('U  in Lattices(ZZ).Hyperbolic():',       U  in Lattices(ZZ).Hyperbolic())
print()

# RootGenerated: a provenance axiom attached by RootLattice, NEVER Gram-detected
A2_root = RootLattice('A2')
print('RootLattice A2 grammar:', A2_root.gram_matrix())
print('In RootGenerated:', A2_root in Lattices(ZZ).Even().RootGenerated())
print('Same grammar without provenance:', Lattice('A2') in Lattices(ZZ).Even().RootGenerated())
assert A2_root.cartan_type() == ('A', 2)

# Negative-definite convention: twist the grammar by -1 (K3 convention)
E8_neg = RootLattice('E', 8, negative=True)
print(f'E₈(-1): sig={E8_neg.signature_pair()}')
assert E8_neg.signature_pair() == (0, 8)  # purely negative

In [ ]:
# Arbitrary grammar: SyntheticLatticeFromGram / Lattice from a matrix
G = matrix(ZZ, 3, 3, [4, 1, 1, 1, 3, 1, 1, 1, 2])
L_custom = Lattice(G, label='custom')
print(f'Custom lattice: {L_custom}')
print(f'  Gram: {L_custom.gram_matrix()}')
print(f'  is_definite={L_custom.is_definite()}, is_positive_definite={L_custom.is_positive_definite()}')
assert L_custom.is_positive_definite()  # det > 0, all minors positive

# Degenerate lattices are valid objects
radical = Lattice(matrix(QQ, 1, 1, [0]), label='rad')
print(f'\nDegenerate: {radical}')
print(f'  is_degenerate={radical.is_degenerate()}, sig={radical.signature_pair()}')
assert radical.is_degenerate()
assert radical not in Lattices(ZZ).Nondegenerate()

## 2. Elements

Lattice elements are coefficient columns in $R^{\mathrm{rank}}$.
The bilinear pairing is `b(v, w) = v^T G w`. Form pairing goes through
`.b()` and `.q()` — never `*`, never `__pow__`.

Elements carry owned module arithmetic (`+`, `-`, scalar `*`).

In [ ]:
U = Lattice('U', label='U')
e, f = U.gens()
print(f'Generators: e={e}, f={f}')
print(f'Rank: {U.rank()}')
print()

# Element arithmetic
print(f'e + f = {e + f}')
print(f'2*e - f = {2*e - f}')
print(f'-(e + f) = {-(e + f)}')

# Form pairing: b = bilinear, q = quadratic (q(v) = b(v,v))
print(f'b(e, f) = {e.b(f)}')      # U: 1
print(f'q(e) = {e.q()}')          # U: 0 (isotropic)
print(f'q(e+f) = {(e+f).q()}')    # U: 2

# Zero element
zero = U.zero()
print(f'zero = {zero}')
print(f'q(zero) = {zero.q()}')

# Random element (for prototyping)
rand = U.random_element()
print(f'random: {rand}, q={rand.q()}')

# maths
assert e + f == U([1, 1])
assert e.b(f) == 1
assert e.q() == 0  # isotropic in hyperbolic plane
assert (e + f).q() == 2

### Noise gating: absent vocabulary

The spike explicitly removes Sage's ambient-space vocabulary.
Form pairing only via `b()`/`q()` — no `*` between elements, no `__pow__`.

In [ ]:
A2 = Lattice('A2', label='A2')
e0 = A2.gen(0)

# Absent: ambient-space vocabulary
for name in ('inner_product', 'dot_product', 'norm', 'hermitian_inner_product',
             'basis_matrix', 'underlying_module', 'rationalization_module'):
    assert not hasattr(e0, name), f'{name} should not resolve'
    assert not hasattr(A2, name), f'{name} should not resolve on parent'
print('Ambient-space vocabulary correctly absent ✓')

## 3. Subobject algebra

Operations between sublattices of a common parent use the underlying module
algebra: `sublattice`, `span`, `overlattice`, `intersection`, `saturation`,
`orthogonal_complement`, `direct_sum`, `tensor_product`.

A subobject is an injective homomorphism into the parent — the child records its
basis in the parent's coordinates, but equality is by $(R, G)$ alone.

In [ ]:
U = Lattice('U', label='U')

# Sublattice from generators in the parent's own basis
diagonal = U.sublattice([[1, 1]], label='diag')
print(f'diagonal sublattice: rank={diagonal.rank()}')
print(f'  Gram: {diagonal.gram_matrix()}')
assert diagonal.gram_matrix() == matrix(QQ, 1, 1, [2])  # b(e+f, e+f) = 2
assert diagonal.is_submodule(U)

# Overlattice: enlarge by rational generators
half_e = matrix(QQ, 1, 2, [QQ(1)/2, 0])
U_half_e = U.overlattice(half_e, label='U_half_e')
print(f'\noverlattice: {U_half_e}')
print(f'  index_in(U) = {U_half_e.index_in(U)}')
assert U.is_submodule(U_half_e)
assert U.index_in(U_half_e) == 2

# Orthogonal complement
e_line = U.sublattice([[1, 0]], label='e')
e_perp = U.orthogonal_complement(e_line)
print(f'\ne^⊥: rank={e_perp.rank()}, Gram={e_perp.gram_matrix()}')
assert e_perp.rank() == 1  # f-line

# Direct sum (block-diagonal grammar)
A2 = Lattice('A2', label='A2')
direct = U.direct_sum(A2, label='U+A2')
print(f'\nU ⊕ A₂: rank={direct.rank()}, sig={direct.signature_pair()}')
assert direct.gram_matrix() == block_diagonal_matrix(U.gram_matrix(), A2.gram_matrix())
assert direct.rank() == 4 and direct.signature_pair() == (3, 1)
# Summands are orthogonal
assert direct.b(direct.gen(0), direct.gen(2)) == 0  # cross-pairing vanishes

In [ ]:
# Intersection, saturation, primitive closure
U = Lattice('U', label='U')
e_line = U.sublattice([[1, 0]], label='e')

# Intersection of sublattices
inter = e_line.intersection(U.sublattice([[2, 0]], label='2e'))
print(f'e ∩ 2e: rank={inter.rank()}, Gram={inter.gram_matrix()}')
assert inter.rank() == 1

# Primitive closure (= saturation in the ambient)
prim = U.sublattice([[2, 0]], label='2e').primitive_closure(U)
print(f'primitive_closure of 2e: rank={prim.rank()}, Gram={prim.gram_matrix()}')
assert prim == e_line  # saturating 2e in U gives e-line

# Index
print(f'index(2e in prim): {U.sublattice([[2, 0]]).index_in(prim)}')
assert U.sublattice([[2, 0]]).index_in(prim) == 2

In [ ]:
# Radical quotient: L / rad(L) → a nondegenerate based lattice
# Stays in Lattices (not a discriminant form)
U = Lattice('U', label='U')
UU = U.direct_sum(U, label='U+U')
e_line = UU.sublattice([[1, 0, 0, 0]], label='e')
e_perp = UU.orthogonal_complement(e_line)
print(f'e^⊥ is_degenerate: {e_perp.is_degenerate()}')

quotient = e_perp.radical_quotient()
print(f'radical_quotient: rank={quotient.rank()}, Gram={quotient.gram_matrix()}')
print(f'  is_nondegenerate: {quotient.is_nondegenerate()}')
assert quotient.is_nondegenerate()
assert quotient.gram_matrix() == U.gram_matrix()  # e^⊥ / e ≅ U
assert quotient in Lattices(ZZ).Nondegenerate()

# Fully degenerate: rad(L) = L → rank 0 quotient
full_degen = Lattice(matrix(ZZ, 2, 2, [0, 0, 0, 0]), label='0^2')
assert full_degen.radical_quotient().rank() == 0

## 4. Dual lattice

For a nondegenerate $\mathbb{Z}$-lattice, the dual has grammar $G^{-1}$.
The metric inclusion $\iota: L \to L^\#$, $v \mapsto b(v, -)$, is represented
by the matrix $G$. Over $\mathbb{Q}$, the dual is the identity.

In [ ]:
A2 = Lattice('A2', label='A2')
A2_dual = A2.dual()
print(f'A₂# Gram: {A2_dual.gram_matrix()}')

# Grammar: G^{-1}
assert A2_dual.gram_matrix() == A2.gram_matrix().inverse()
# Involutive: dual of dual = original
assert A2_dual.dual() == A2

# The metric inclusion L → L#: v ↦ b(v, -), represented by G
inclusion = A2.dual_inclusion()
print(f'dual_inclusion: domain={inclusion.domain()}, codomain={inclusion.codomain()}')
assert inclusion.codomain() == A2_dual
assert inclusion.image().is_submodule(A2_dual)

# Over QQ: dual is the identity (metric dual ≡ space)
V = A2.rationalization()
print(f'\nRationalization: {V}')
assert V.base_ring() is QQ
assert V.dual() is V  # ratified: dual = self over QQ

## 5. Discriminant group

For an integral nondegenerate lattice $L$, the discriminant group
$A_L = L^\# / L$ is a finite abelian group carrying two torsion forms:
the **bilinear** form $b \colon A_L \times A_L \to \mathbb{Q}/\mathbb{Z}$
and the **quadratic** refinement $q \colon A_L \to \mathbb{Q}/2\mathbb{Z}$
with $q(x) \equiv b(x,x) \pmod{\mathbb{Z}}$ (Nikulin convention).

The discriminant group is typed in `DiscriminantForms(ZZ)` — NOT in
`Lattices(ZZ)`.

In [ ]:
A2 = Lattice('A2', label='A2')
D = A2.discriminant_group()
print(f'Discriminant group of A₂: {D}')
print(f'  invariants: {D.invariants()}')
print(f'  cardinality: {D.cardinality()}')
print()

# Facts about A₂'s discriminant group
assert D.invariants() == (3,)       # A₂#/A₂ ≅ ℤ/3ℤ
assert D.cardinality() == 3

# Category: discriminant group lives in DiscriminantForms, not Lattices
assert D in DiscriminantForms(ZZ)
assert D not in Lattices(ZZ)

# Form values
g = D.gen(0)
print(f'generator: {g}')
print(f'  q(g) = {g.q()}  (value in QQ/2ZZ)')
print(f'  b(g, g) = {D.b(g, g)}  (value in QQ/ZZ)')
assert g.q() == QQ(2)/3  # for A₂
assert g + 2*g == D([0])  # order 3 element
assert 3*g == D.zero()

# Lift back to the lattice (into QQ-span)
lifted = D.lift(g)
print(f'\nlift to L⊗QQ: q={lifted.q()}')
assert D.projection(lifted) == g  # projection-lift round-trip

In [ ]:
# Subgroups, isotropic subgroups, overlattice construction
L = Lattice(matrix(ZZ, 1, 1, [4]), label='⟨4⟩')
D = L.discriminant_group()
g = D.gen(0)

# Subgroup generated by 2g
H = D.subgroup_generated_by([2 * g])
print(f'Subgroup <2g>: |H| = {H.cardinality()}')
assert H.cardinality() == 2

# Isotropy
print(f'  is_bilinear_isotropic: {H.is_bilinear_isotropic()}')
print(f'  is_quadratic_isotropic: {H.is_quadratic_isotropic()}')
assert H.is_bilinear_isotropic()
assert not H.is_quadratic_isotropic()  # q(2g) = 0 mod 2ZZ (even) but not in the quadratic sense

# Orthogonal complement in discriminant group
Hperp = D.orthogonal_submodule_to(H)
print(f'  |H^⊥| = {Hperp.cardinality()}')

# Overlattice from isotropic subgroup (Nikulin gluing)
M = D.overlattice_from_isotropic_subgroup(H, label='⟨1⟩')
print(f'\nOverlattice from H: Gram = {M.gram_matrix()}')
assert M.gram_matrix() == matrix(QQ, 1, 1, [1])  # ⟨4⟩ enlarged by 2g → ⟨1⟩

## 6. Genus and Nikulin theory

The genus $\mathrm{gen}(L)$ is the isomorphism class of the discriminant form
together with the signature pair. Nikulin's theorem: for an isotropic subgroup
$H \subset A_L$, the overlattice's discriminant form is isometric to
$H^\perp / H$. The gluing construction that applies this theorem appears in
§12 (`IntegralLatticeGluing`).

In [ ]:
A2 = Lattice('A2', label='A2')
D = A2.discriminant_group()

# Genus: discriminant form + signature pair + evenness flag
gen = D.genus((2, 0))  # sig = (2, 0), note: the form is already even
print(f'Genus of A₂: {gen}')
assert isinstance(gen, SyntheticGenus)
assert A2.genus() == gen
assert D.is_genus((2, 0))  # the discriminant form carries the correct genus data

# Brown invariant (for even forms)
if D.brown_invariant() is not None:
    print(f'Brown invariant: {D.brown_invariant()}')  # A₂: 2

# Same genus check
assert A2.same_genus(A2)
assert A2.genus() == A2.genus()

In [ ]:
# Nikulin theorem (1980): for isotropic H ⊂ A_L,
#   A_{L'} ≅ H^⊥/H  (form isometry)
L = Lattice(matrix(ZZ, 3, 3, [0, 2, 0, 2, 0, 0, 0, 0, 2]), label='U2+A1')
D = L.discriminant_group()
print(f'Discriminant form invariants: {D.invariants()}')
assert D.invariants() == (2, 2, 2)

for H in D.isotropic_subgroups():
    if H.cardinality() == 1:
        continue  # trivial subgroup
    overlattice_form = D.discriminant_form_of_overlattice(H)
    quotient_form = D.orthogonal_quotient(H)  # H^⊥ / H
    # Both sides, presented as quadratic forms, must be isometric
    assert overlattice_form.is_isomorphic(quotient_form, kind='quadratic')
    # Miranda-Morrison normal form cross-check
    assert overlattice_form.miranda_morrison_normal_form() == quotient_form.miranda_morrison_normal_form()
print('Nikulin theorem verified for all non-trivial isotropic subgroups ✓')

In [ ]:
# Metabolizer: a Lagrangian = self-orthogonal isotropic subgroup H = H^⊥
# U(2) is metabolic
L_u2 = Lattice(matrix(ZZ, 2, 2, [0, 2, 2, 0]), label='U(2)')
A = L_u2.discriminant_group()
print(f'U(2) discriminant invariants: {A.invariants()}')

assert A.is_metabolic()
H = A.metabolizer()
# q|_H = 0 (isotropic)
assert all(A.q(h) == 0 for h in H.elements())
# H = H^⊥ (Lagrangian property)
assert set(A.orthogonal(H).elements()) == set(H.elements())
print(f'Metabolizer: |H| = {H.cardinality()}, H = H^⊥ ✓')

## 7. Homsets and morphisms

`L.Hom(M)` returns a form-preserving homset. Morphisms are constructed
via `.from_matrix()` and are form-preserving by construction.

Operations: `kernel()`, `image()` (as synthetic lattices),
`is_isometry()`, composition, reflection.

In [ ]:
U = Lattice('U', label='U')

# Homset construction
Hom_U = U.Hom(U)
print(f'Homset: {Hom_U}')

# Identity morphism
id_U = Hom_U.from_matrix(identity_matrix(ZZ, 2))
assert id_U(U.gen(0)) == U.gen(0)
assert id_U(U.gen(0)).b(id_U(U.gen(1))) == U.gen(0).b(U.gen(1))  # form-preserving

# Swap morphism
swap = Hom_U.from_matrix(matrix(ZZ, 2, 2, [0, 1, 1, 0]))
print(f'swap(e) = {swap(U.gen(0))}')
print(f'swap(f) = {swap(U.gen(1))}')
assert swap(U.gen(0)) == U.gen(1)
assert swap(U.gen(1)) == U.gen(0)

# Composition: swap * swap = identity
assert (swap * swap)(U.gen(0)) == U.gen(0)

In [ ]:
# Reflection: σ_v(x) = x - (2 b(x,v)/q(v)) v
# Defined only for anisotropic vectors (q(v) ≠ 0)
A2 = Lattice('A2', label='A2')
e0, e1 = A2.gen(0), A2.gen(1)

sigma = A2.reflection(e0)
print(f'Reflection in e0:')
print(f'  σ(e0) = {sigma(e0)}  (should be -e0)')
print(f'  σ(e1) = {sigma(e1)}')
print(f'  σ²(e0) = {sigma(sigma(e0))}  (should be e0, order 2)')
print(f'  σ²(e1) = {sigma(sigma(e1))}')

assert sigma(e0) == -e0
assert sigma(sigma(e0)) == e0  # order 2
assert sigma(sigma(e1)) == e1
# Form-preserving
assert A2.q(sigma(e1)) == A2.q(e1)
assert A2.b(sigma(e0), sigma(e1)) == A2.b(e0, e1)

## 8. Isometry group $O(L)$

`L.isometry_group()` returns $O(L)$ — total for every lattice, unique per
lattice. For definite lattices, generators are computed lazily.
Elements are endomorphism objects with composition, inversion, and order.

In [ ]:
A2 = Lattice('A2', label='A2')
O = A2.isometry_group()
print(f'O(A₂): finite={O.is_finite()}, order={O.order()}')
assert O.is_finite() and O.order() == 12  # O(A₂) ≅ W(A₂) × {±I} ≅ S₃ × C₂ (order 12)

# Generators: computed lazily, always form-preserving
gens = O.gens()
print(f'\nGenerators: {len(list(gens))}')
for g in gens:
    m = g.matrix()
    print(f'  matrix: {m}, det={m.det()}')
    assert m.transpose() * A2.gram_matrix() * m == A2.gram_matrix()

# Group algebra via morphism composition
s = A2.reflection(A2.gen(0))
t = A2.reflection(A2.gen(1))
assert s in O and t in O

c = s * t  # Coxeter element
print(f'\nCoxeter element c = s∘t: order={c.order()}')
assert c.order() == 3  # h(A2) = 3

# Inversion, powers
assert s * s == O.one()
assert s ** (-1) == s
assert s.inverse() == s

# Unipotent predicate
print(f's.is_unipotent(): {s.is_unipotent()}')   # false
print(f'id.is_unipotent(): {O.one().is_unipotent()}')  # true

In [ ]:
# O(L) action on discriminant group: functorial
U = Lattice('U', label='U')
O_U = U.isometry_group()
swap = O_U.from_matrix(matrix(ZZ, 2, 2, [0, 1, 1, 0]))
assert swap(U.gen(0)) == U.gen(1)

# Discriminant group of U is trivial (unimodular) → action is identity
assert U.isometry_group().discriminant_action(swap).is_identity()

# Subgroup discriminant image
assert U.isometry_group().subgroup([matrix(ZZ, 2, 2, [0, 1, 1, 0])]).discriminant_image()[0].is_identity()

## 9. Isometry detection

`L.is_isometric(M)` decides isometry between two synthetic lattices
through Sage's own engines (rational equivalence over $\mathbb{Q}$,
global equivalence for definite forms, genus + spinor-genus theory for
indefinite rank ≥3).

In [ ]:
# Definite: decided by PARI qfisom
A2 = Lattice('A2', label='A2')
assert A2.is_isometric(Lattice('A2', label='other'))  # same grammar
assert not A2.is_isometric(Lattice(matrix(ZZ, 2, 2, [2, 0, 0, 2])))  # 2I ≠ A2

# Rational equivalence
half = Lattice(matrix(QQ, [[QQ(1)/2]]), base_ring=QQ, label='half')
two  = Lattice(matrix(QQ, [[2]]), base_ring=QQ, label='two')
assert half.is_isometric(two)  # ratio 4 = square

# Unimodular transforms preserve isometry class
G = matrix.diagonal(ZZ, [1, 1, -3, -5])
L = Lattice(G, label='L')
T = matrix(ZZ, 4, 4, [1, 1, 0, 0, 0, 1, 0, 0, 2, 0, 1, 1, 1, 0, 0, 1])
assert T.det() in (1, -1)
assert L.is_isometric(Lattice(T * G * T.transpose(), label='M'))

# Degenerate: recurse via radical quotient
assert Lattice(matrix(ZZ, 2, 2, [0, 0, 0, 2])).is_isometric(Lattice(matrix(ZZ, 2, 2, [2, 2, 2, 2])))
print('Isometry detection verified ✓')

## 10. Positive-definite enumeration

The reduction/CVP/Voronoi suite lives on the positive-definite kernel:
`short_vectors`, `shortest_vector`, `closest_vector`, `voronoi_cell`,
`BKZ`, `HKZ`, `gaussian_heuristic`, `hadamard_ratio`.

Absent off the positive-definite subcategory (category gating).

In [ ]:
A2 = Lattice('A2', label='A2')

# Vectors of fixed square
roots = A2.vectors_of_square(2)
print('Vectors of square 2 (roots):')
for v in roots:
    print(f'  {tuple(v.coefficient_vector())}, q={v.q()}')
assert len(roots) == 6  # A₂ has 6 roots
assert all(v.q() == 2 for v in roots)

# Shortest vector
sv = A2.shortest_vector()
print(f'\nShortest vector: {sv}, q={sv.q()}')
assert sv.q() == 2  # λ₁² of A₂

# Minimum
print(f'Minimum λ₁²: {A2.minimum()}')
assert A2.minimum() == 2

In [ ]:
# Closest vector problem
A2 = Lattice('A2', label='A2')
target = [QQ(2)/3, QQ(2)/3]
cvp = A2.closest_vector(target)
print(f'CVP to {target}: {tuple(cvp.coefficient_vector())}')
assert tuple(cvp.coefficient_vector()) == (1, 1)

# Babai approximate CVP
approx = A2.approximate_closest_vector(target)
print(f'Babai CVP: {tuple(approx.coefficient_vector())}')

# Reduced basis (LLL-reduced in lattice's own coordinates)
basis = A2.reduced_basis()
print(f'\nReduced basis Gram: {matrix(ZZ, [[b.coefficient_vector()[i] for i in range(2)] for b in basis])}')

# Volume = sqrt(det)
print(f'\nVolume: {A2.volume()}')
assert abs(A2.volume() - A2.gram_matrix().determinant().sqrt()) < 1e-12

# Gaussian heuristic
print(f'Gaussian heuristic: {A2.gaussian_heuristic()}')
print(f'Exact form:          {A2.gaussian_heuristic(exact_form=True)}')

# Hadamard ratio
print(f'Hadamard ratio: {A2.hadamard_ratio()}')

# Category gating: indefinite lattices have none of this
U = Lattice('U', label='U')
for name in ('minimum', 'maximum', 'volume', 'short_vectors', 'closest_vector',
             'voronoi_cell', 'BKZ', 'LLL', 'gaussian_heuristic'):
    assert not hasattr(U, name), f'{name} absent on indefinite ✓'
print('PD enumeration suite correctly gated from indefinite lattice ✓')

In [ ]:
# Voronoi cell
A2 = Lattice('A2', label='A2')
V = A2.voronoi_cell()
print(f'Voronoi cell vertices: {len(V.vertices())}')
assert len(V.vertices()) == 6  # A₂ Voronoi cell is a hexagon

# Voronoi relevant vectors
relevant = A2.voronoi_relevant_vectors()
print(f'Voronoi relevant vectors: {len(relevant)}')
assert len(relevant) == 6  # all 6 roots are Voronoi-relevant for A₂
assert all(v.q() == 2 for v in relevant)

# BKZ reduction
L = Lattice(matrix(ZZ, 3, 3, [4, 1, 1, 1, 3, 1, 1, 1, 2]), label='G')
bkz = L.BKZ(block_size=2)
print(f'\nBKZ-reduced Gram: {bkz.gram_matrix()}')
assert bkz.determinant() == L.determinant()  # covolume preserved

## 11. Negative-definite enumeration

The negative-definite kernel transports through $L(-1)$:
short vectors of a negative-definite $L$ are the short vectors of $L(-1)$.
Roots of a negative-definite lattice are its $(-2)$-vectors (AG convention).

In [ ]:
A2 = Lattice('A2', label='A2')
A2_neg = A2.twist(-1)
assert A2_neg.is_negative_definite()

print(f'A₂(-1) signature: {A2_neg.signature_pair()}')
print(f'minimum (λ₁²): {A2_neg.minimum()}')   # -∞
print(f'maximum: {A2_neg.maximum()}')          # -2

# Shortest vector: a root of the negative-definite lattice
sv_neg = A2_neg.shortest_vector()
print(f'shortest vector: {sv_neg}, q={sv_neg.q()}')
assert sv_neg.q() == -2  # (-2)-vector, the AG root convention

# Roots of the negative-definite lattice = (-2)-vectors
neg_roots = A2_neg.roots()
print(f'Roots: {len(neg_roots)} vectors of square -2')
assert all(r.q() == -2 for r in neg_roots)

# Bijection with positive-definite roots: same coefficient vectors
assert len(neg_roots) == len(A2.roots())

# Volume preserved under sign twist
assert A2_neg.volume() == A2.volume()

## 12. IntegralLatticeGluing

Construct an overlattice from discriminant glue vectors.
Each glue vector provides one discriminant element per source lattice.

In [ ]:
# Glue two copies of ⟨4⟩ via the isotropic subgroup 2g + 2g
L = Lattice(matrix(ZZ, 1, 1, [4]), label='⟨4⟩')
g = L.discriminant_group().gen(0)
glued = IntegralLatticeGluing([L, L], [[2*g, 2*g]], label='glued ⟨4⟩⊕⟨4⟩')
print(f'Glued lattice: rank={glued.rank()}')
print(f'  Gram: {glued.gram_matrix()}')
assert glued.rank() == 2
print('Gluing construction works ✓')

## 13. Standalone discriminant forms

Discriminant forms can be constructed standalone (not sourced from a lattice):
`TorsionQuadraticForm` and `SyntheticBilinearDiscriminantForm`.

In [ ]:
# Standalone quadratic discriminant form
D = TorsionQuadraticForm(matrix.diagonal(QQ, [QQ(1)/2]))
z = D.gen(0)
print(f'Standalone TQF: invariants={D.invariants()}')
print(f'  q(z) = {D.q(z)}')
print(f'  b(z, z) = {D.b(z, z)}')
# The quadratic form refines the diagonal: q(x) = b(x, x) mod ZZ
assert (D.q(z) - D.b(z, z)) in ZZ

# Standalone bilinear form (induces its diagonal quadratic form)
B = SyntheticBilinearDiscriminantForm(matrix.diagonal(QQ, [QQ(1)/2, QQ(1)/3]))
x, y = B.gen(0), B.gen(1)
print(f'\nBilinear form: invariants={B.invariants()}')
print(f'  q(x) = b(x,x) = {B.q(x)}')
assert B.q(x) == B.b(x, x)
assert B.q(x + y) == B.b(x + y, x + y)
print('Bilinear → quadratic induced ✓')

## 14. Axiom gating

Vocabulary placement by category membership: an operation exists exactly
where the mathematics defines it. Dual needs nondegeneracy; genus needs
integral nondegenerate; enumeration suite needs positive-definite (or its
negative-definite transport).

Absence (where operation is mathematically undefined) → name does not
resolve. It does NOT raise.

In [ ]:
A2 = Lattice('A2', label='A2')
degenerate = Lattice(matrix(QQ, [[0, 0], [0, 0]]), label='deg')
rational = Lattice(matrix(QQ, [[QQ(1)/2]]), base_ring=QQ, label='half')

# Axis 1 (definedness → placement/absence)
for name in ('dual', 'dual_inclusion', 'discriminant_group', 'genus', 'same_genus',
             'glue', 'maximal_overlattice', 'local_modification'):
    assert hasattr(A2, name),           f'{name} present on integral nondegenerate'
    assert not hasattr(degenerate, name), f'{name} absent on degenerate'

for name in ('discriminant_group', 'genus', 'same_genus', 'glue'):
    assert not hasattr(rational, name), f'{name} absent on rational (QQ) lattice'

assert hasattr(rational, 'dual')  # dual exists on nondegenerate, even over QQ

# Axis 2 (enumeration gating)
for name in ('minimum', 'short_vectors', 'shortest_vector', 'closest_vector'):
    assert hasattr(A2, name),           f'{name} present on positive-definite'
    assert not hasattr(degenerate, name), f'{name} absent on degenerate'

U = Lattice('U', label='U')
for name in ('minimum', 'short_vectors', 'BKZ', 'LLL', 'voronoi_cell'):
    assert not hasattr(U, name), f'{name} absent on indefinite ✓'

print('Axiom gating verified across all specimens ✓')

## Summary

The base parity spike provides:

- **One clean API**: `Lattices(R).from_gram_matrix()` is the only entry point
- **Category gating**: vocabulary resolves exactly where mathematics defines it
- **No ambient space**: based model — $(R, G)$ is the identity
- **Owned discriminant forms**: finite abelian groups with bilinear/quadratic forms
- **Form-preserving homsets**: kernel, image, isometry detection
- **Isometry group $O(L)$**: total, with group algebra
- **Reduction/CVP/Voronoi suite**: on the positive-definite kernel
- **Negative-definite transport**: through $L(-1)$
- **Nikulin theory**: overlattice from isotropic subgroups

For the full specification, see `SYNTHETIC_LATTICE_MODEL.md`. For literature
fixtures and Sage reference contracts, see `tests/` (run via `just test`
from this directory).